# Dashboard Comercial y Operativo — Café Monte L

**Autor**: Juan Diego Argüello Nájera  
**Herramientas**: Python · pandas · NumPy · Plotly · ipywidgets · fpdf2  
**Dataset**: Datos propios (6 archivos CSV: Ventas, Productos, Clientes, Presupuesto, Producción, Marketing)

---

## Objetivo

Construir un dashboard interactivo que permita al equipo comercial monitorear ventas vs. presupuesto
y al equipo operativo analizar la producción por planta y turno, todo desde un solo notebook.

## Preguntas de negocio

1. ¿Qué regiones y canales están por debajo del presupuesto de ventas?
2. ¿Qué familias de producto tienen el margen más débil?
3. ¿Cómo se comporta la producción por planta, línea y turno?

## Estructura del notebook

1. Instalación de librerías y carga de datos  
2. Construcción de tabla maestra (merge de 6 fuentes)  
3. Tablas dinámicas: Ventas vs. Presupuesto por Región y Canal  
4. Dashboard Comercial con segmentadores interactivos  
5. Tablas dinámicas operativas (producción por turno)  
6. Dashboard Operativo con KPIs de planta  
7. Exportación a PDF  

---
> **Nota**: Para ejecutar localmente, descarga los 6 archivos CSV desde `/data/`
> y asegúrate de instalar las dependencias con `pip install -r requirements.txt`

In [ ]:
# Instalación de librerías necesarias
!pip install plotly kaleido fpdf2 Pillow ipywidgets --quiet

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from fpdf import FPDF
import os, warnings

warnings.filterwarnings('ignore')
print('✅ Librerías cargadas correctamente.')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.0/81.0 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.1/327.1 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.3/49.3 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 41.9 MB/s eta 0:00:00
✅ Librerías cargadas correctamente.


In [ ]:
from google.colab import files
print("Por favor, sube los 6 archivos CSV (Ventas, Productos, Clientes, Presupuesto, Producción, Marketing):")
uploaded = files.upload()

def clean_numeric(serie):
    return pd.to_numeric(serie.astype(str).str.replace(r'[$,%]', '', regex=True).str.replace(',', ''), errors='coerce')

# Carga con codificación utf-8-sig para evitar errores con caracteres invisibles (BOM)
df_ventas = pd.read_csv('Ventas_Detalle_Raw.csv', encoding='utf-8-sig')
df_prod_cat = pd.read_csv('Catalogo_Productos.csv', encoding='utf-8-sig')
df_clie_cat = pd.read_csv('Catalogo_Clientes.csv', encoding='utf-8-sig')
df_presupuesto = pd.read_csv('Presupuesto_Mensual.csv', encoding='utf-8-sig')
df_produccion = pd.read_csv('Produccion_Lotes_Raw.csv', encoding='utf-8-sig')



# ── Rutas relativas (funciona en cualquier equipo) ──────────────────────────
'''DATA_DIR = os.path.join(os.path.dirname(os.path.abspath("__file__")), "data")
import os

df_ventas    = pd.read_csv(os.path.join(DATA_DIR, "Ventas_Detalle_Raw.csv"),        encoding="utf-8-sig")
df_prod_cat  = pd.read_csv(os.path.join(DATA_DIR, "Catalogo_Productos.csv"),        encoding="utf-8-sig")
df_clie_cat  = pd.read_csv(os.path.join(DATA_DIR, "Catalogo_Clientes.csv"),         encoding="utf-8-sig")
df_presupuesto = pd.read_csv(os.path.join(DATA_DIR, "Presupuesto.csv"),             encoding="utf-8-sig")
df_produccion  = pd.read_csv(os.path.join(DATA_DIR, "Produccion.csv"),              encoding="utf-8-sig")
df_marketing   = pd.read_csv(os.path.join(DATA_DIR, "Marketing.csv"),               encoding="utf-8-sig")

print("✅ Datos cargados correctamente.")'''

# --- LIMPIEZA OBLIGATORIA (Según Guía de Depuración) ---
for df in [df_ventas, df_prod_cat, df_clie_cat, df_presupuesto, df_produccion]:
    if 'SKU' in df.columns: df['SKU'] = df['SKU'].str.strip().str.upper()
    if 'ID_Cliente' in df.columns: df['ID_Cliente'] = df['ID_Cliente'].str.strip().str.upper()
    if 'Region' in df.columns: df['Region'] = df['Region'].str.strip()

# Normalización de fechas y creación de columna "Mes" para cruces
df_ventas['Fecha_Pedido'] = pd.to_datetime(df_ventas['Fecha_Pedido'])
df_ventas['Mes_Nombre'] = df_ventas['Fecha_Pedido'].dt.strftime('%b').str.lower() # sep, oct, nov, dic

df_presupuesto['Mes_Dt'] = pd.to_datetime(df_presupuesto['Mes'])
df_presupuesto['Mes_Nombre'] = df_presupuesto['Mes_Dt'].dt.strftime('%b').str.lower()

# Limpieza de columnas numéricas en Ventas
cols_money = ['Ingreso_Bruto', 'Ingreso_Neto', 'Costo_Logistico', 'Precio_Lista', 'Precio_Final']
for c in cols_money: df_ventas[c] = clean_numeric(df_ventas[c])

print('✅ Datos cargados y depurados.')

Por favor, sube los 6 archivos CSV (Ventas, Productos, Clientes, Presupuesto, Producción, Marketing):


Saving Campanas_Marketing_Raw.csv to Campanas_Marketing_Raw.csv
Saving Catalogo_Clientes.csv to Catalogo_Clientes.csv
Saving Catalogo_Productos.csv to Catalogo_Productos.csv
Saving Presupuesto_Mensual.csv to Presupuesto_Mensual.csv
Saving Produccion_Lotes_Raw.csv to Produccion_Lotes_Raw.csv
Saving Ventas_Detalle_Raw.csv to Ventas_Detalle_Raw.csv
✅ Datos cargados y depurados.


In [33]:
# 1. Unir Ventas con Catálogos
df_maestra = pd.merge(df_ventas, df_prod_cat[['SKU', 'Familia', 'Costo_Estandar_Unitario', 'Margen_Objetivo']], on='SKU', how='left')
df_maestra = pd.merge(df_maestra, df_clie_cat[['ID_Cliente', 'Segmento_Cliente', 'Clasificacion']], on='ID_Cliente', how='left')

# 2. Unir con Presupuesto (Cruce por Mes, Region, Canal y Familia)
df_maestra = pd.merge(df_maestra, df_presupuesto[['Mes_Nombre', 'Region', 'Canal', 'Familia', 'Presupuesto_Ventas', 'Presupuesto_Margen']],
                     on=['Mes_Nombre', 'Region', 'Canal', 'Familia'], how='left')

# 3. CÁLCULOS ESPECÍFICOS (Según Guía Tabla Maestra)
df_maestra['Costo_Estandar_Total'] = df_maestra['Unidades'] * df_maestra['Costo_Estandar_Unitario']
df_maestra['Margen_Estimado'] = df_maestra['Ingreso_Neto'] - df_maestra['Costo_Estandar_Total'] - df_maestra['Costo_Logistico']
df_maestra['Margen_Pct'] = (df_maestra['Margen_Estimado'] / df_maestra['Ingreso_Neto']).fillna(0)
df_maestra['Descuento_Importe'] = df_maestra['Ingreso_Bruto'] - df_maestra['Ingreso_Neto']

# 4. Preparar Tabla de Producción para KPIs operativos
df_produccion['Merma_Pct'] = df_produccion['Unidades_Merma'] / df_produccion['Unidades_Producidas']
df_produccion['Costo_Total_Lote'] = df_produccion['Costo_Materia_Prima'] + df_produccion['Costo_Mano_Obra'] + df_produccion['Costo_Indirecto']

print('✅ Tabla Maestra generada con campos calculados.')

✅ Tabla Maestra generada con campos calculados.


In [34]:
# --- 1. VISUALIZACIÓN DE LA TABLA MAESTRA ---
display(HTML("<h2>Tabla Maestra Comercial-Financiera (Primeros 5 registros)</h2>"))
display(df_maestra.head(40))

# --- 2. TABLAS DINÁMICAS (Resúmenes Analíticos solicitados en PDF) ---

# A. Tabla Dinámica de Ventas vs Presupuesto por Región y Canal
# Nota: Usamos 'mean' para el presupuesto porque el valor se repite en las filas de la tabla maestra
pivot_presupuesto = df_maestra.groupby(['Region', 'Canal']).agg({
    'Ingreso_Neto': 'sum',
    'Presupuesto_Ventas': 'sum' # Al ser valores por grupo, sumamos el presupuesto asignado
}).rename(columns={'Ingreso_Neto': 'Ventas Reales', 'Presupuesto_Ventas': 'Meta Ventas'})

pivot_presupuesto['Cumplimiento %'] = (pivot_presupuesto['Ventas Reales'] / pivot_presupuesto['Meta Ventas'])

# B. Tabla Dinámica de Rentabilidad por Familia y Segmento de Cliente
pivot_rentabilidad = pd.pivot_table(
    df_maestra,
    values='Margen_Pct',
    index=['Familia'],
    columns=['Segmento_Cliente'],
    aggfunc='mean'
)

# C. Tabla Dinámica de Desempeño Operativo (Producción)
pivot_operativa = df_produccion.groupby(['Planta', 'Linea']).agg({
    'Unidades_Producidas': 'sum',
    'Merma_Pct': 'mean',
    'Tiempo_Muerto_Min': 'sum'
}).rename(columns={'Merma_Pct': '% Merma Promedio', 'Tiempo_Muerto_Min': 'Total Tiempo Muerto'})

# --- 3. MOSTRAR TABLAS DINÁMICAS EN PANTALLA ---

display(HTML("<h3>Tabla Dinámica 1: Cumplimiento de Ventas por Región y Canal</h3>"))
display(pivot_presupuesto.style.format({'Ventas Reales': '${:,.2f}', 'Meta Ventas': '${:,.2f}', 'Cumplimiento %': '{:.2%}'}))

display(HTML("<h3>Tabla Dinámica 2: Margen % Promedio por Familia y Segmento</h3>"))
display(pivot_rentabilidad.style.format('{:.2%}'))

display(HTML("<h3>Tabla Dinámica 3: Eficiencia Operativa por Planta y Línea</h3>"))
display(pivot_operativa.style.format({'Unidades_Producidas': '{:,.0f}', '% Merma Promedio': '{:.2%}', 'Total Tiempo Muerto': '{:,.0f} min'}))

,ID_Pedido,Fecha_Pedido,Fecha_Entrega,ID_Cliente,Cliente,Ciudad,Region,Canal,Vendedor,SKU,...,Costo_Estandar_Unitario,Margen_Objetivo,Segmento_Cliente,Clasificacion,Presupuesto_Ventas,Presupuesto_Margen,Costo_Estandar_Total,Margen_Estimado,Margen_Pct,Descuento_Importe
0,PED010000,2025-10-21,2025-10-24,CL044,Café Ruta 11,Puebla,Centro,Autoservicio,Andrés Castañeda,SKU013,...,92,0.40,Distribuidor regional,B,NaN,NaN,21160,10763.32,0.307531,5480.90
1,PED010000,2025-10-27,2025-10-31,CL013,Mayoreo Sol del Pacífico MX,Torreón,Norte,Autoservicio,Paola Luna,SKU002,...,82,0.32,Minorista independiente,A,369590.0,85410.0,7216,3051.28,0.266106,1557.60
2,PED010001,2025-11-21,2025-11-24,CL019,Café y Punto Gourmet Comercial,Hermosillo,Norte,E-commerce,Andrés Castañeda,SKU003,...,164,0.32,Cafetería premium,A,287120.0,68400.0,4920,1614.56,0.204013,366.00
3,PED010001,2025-11-13,2025-11-16,CL014,Comercializadora Horizonte Comercial,Saltillo,Norte,E-commerce,Valeria Solís,SKU005,...,126,0.38,Distribuidor regional,B,234640.0,74250.0,252,115.82,0.274039,25.36
4,PED010002,2025-11-19,2025-11-22,CL029,Tiendas Urbanas MX,Ciudad de México,Centro,Autoservicio,Sofía Gil,SKU003,...,164,0.32,Marketplace,B,412410.0,95320.0,33620,12983.07,0.256270,5918.35
5,PED010002,2025-12-13,2025-12-16,CL033,Mayoreo Sol del Pacífico Comercial,Querétaro,Centro,HORECA,Valeria Solís,SKU004,...,61,0.38,Distribuidor regional,B,265930.0,81450.0,3965,2029.36,0.296411,823.55
6,PED010003,2025-10-18,2025-10-24,CL069,Tiendas Urbanas MX S.A. de C.V.,Morelia,Occidente,HORECA,Iván Durán,SKU014,...,238,0.40,Marketplace,A,NaN,NaN,26418,9088.21,0.222894,3182.37
7,PED010003,2025-09-15,2025-09-21,CL039,Café y Punto Gourmet,Toluca,Centro,E-commerce,Iván Durán,SKU002,...,82,0.32,Cadena nacional,B,261230.0,61850.0,2624,1277.64,0.273486,64.32
8,PED010004,2025-12-30,2026-01-02,CL013,Mayoreo Sol del Pacífico MX,Torreón,Norte,HORECA,Andrés Castañeda,SKU018,...,89,0.37,Minorista independiente,A,NaN,NaN,5518,2440.07,0.265685,1231.94
9,PED010004,2025-11-16,2025-11-21,CL012,Tiendas Plaza Viva S. de R.L.,Monterrey,Norte,HORECA,Sofía Gil,SKU019,...,84,0.39,Distribuidor regional,A,NaN,NaN,1848,938.14,0.293319,277.64


Segmento_Cliente,Cadena nacional,Cafetería premium,Corporativo,Distribuidor regional,Marketplace,Minorista independiente
Familia,,,,,,
Bebidas listas,30.55%,32.38%,30.76%,32.26%,31.31%,30.51%
Cápsulas,39.00%,39.01%,37.62%,38.88%,39.03%,38.15%
Descafeinado,27.19%,26.52%,26.67%,27.41%,26.23%,25.63%
Especialidad,27.61%,26.90%,28.88%,27.23%,27.95%,27.32%
Orgánico,25.30%,26.01%,26.76%,26.88%,25.99%,26.20%
Premium,29.25%,28.65%,27.57%,28.37%,28.55%,28.49%
Tradicional,28.05%,27.78%,26.85%,28.26%,27.75%,28.00%


In [35]:
# --- 1. DEFINICIÓN DE SEGMENTADORES ---
import ipywidgets as widgets
from IPython.display import display, HTML

# Variables globales para que la siguiente celda las pueda leer
filtros_activos = {
    'regiones': list(df_maestra['Region'].unique()),
    'canales': list(df_maestra['Canal'].unique()),
    'familias': list(df_maestra['Familia'].unique())
}

sel_region = widgets.SelectMultiple(
    options=sorted(df_maestra['Region'].unique()),
    value=tuple(df_maestra['Region'].unique()),
    description='📍 Regiones:',
    style={'description_width': 'initial'}
)

sel_canal = widgets.SelectMultiple(
    options=sorted(df_maestra['Canal'].unique()),
    value=tuple(df_maestra['Canal'].unique()),
    description='🛒 Canales:',
    style={'description_width': 'initial'}
)

sel_familia = widgets.SelectMultiple(
    options=sorted(df_maestra['Familia'].unique()),
    value=tuple(df_maestra['Familia'].unique()),
    description='☕ Familias:',
    style={'description_width': 'initial'}
)

btn_confirmar = widgets.Button(
    description="✅ Guardar Filtros y Preparar Dashboard",
    button_style='info',
    layout={'width': '98%', 'margin': '10px 0px'}
)

def guardar_filtros(b):
    filtros_activos['regiones'] = list(sel_region.value)
    filtros_activos['canales'] = list(sel_canal.value)
    filtros_activos['familias'] = list(sel_familia.value)
    print("✅ Filtros guardados. Ahora ejecuta la celda de abajo para ver los resultados.")

btn_confirmar.on_click(guardar_filtros)

display(HTML("<h3>1. Selecciona tus filtros (Usa Ctrl+Click para varios)</h3>"))
display(widgets.HBox([sel_region, sel_canal, sel_familia]))
display(btn_confirmar)

Button(button_style='info', description='✅ Guardar Filtros y Preparar Dashboard', layout=Layout(margin='10px 0…

In [36]:
# --- LÓGICA DE VISUALIZACIÓN ---
from IPython.display import clear_output

# 1. Filtrar los DataFrames según la selección de la celda anterior
df_f = df_maestra[
    (df_maestra['Region'].isin(filtros_activos['regiones'])) &
    (df_maestra['Canal'].isin(filtros_activos['canales'])) &
    (df_maestra['Familia'].isin(filtros_activos['familias']))
]

# Plantas para el área operativa
plantas_sel = [f"Planta {r}" for r in filtros_activos['regiones']]
dp_f = df_produccion[df_produccion['Planta'].isin(plantas_sel)]

if df_f.empty:
    print("⚠️ No hay datos para la selección actual.")
else:
    # --- CÁLCULO DE INDICADORES ---
    ing_neto = df_f['Ingreso_Neto'].sum()
    marg_est = df_f['Margen_Estimado'].sum()
    marg_pct = (marg_est / ing_neto) if ing_neto != 0 else 0

    entregas = df_f[df_f['Estado_Entrega'] != 'Pendiente']
    tasa_ent = (entregas['Estado_Entrega'] == 'Entregado a tiempo').mean() if not entregas.empty else 0

    # --- MOSTRAR KPIs (FONDO CLARO) ---
    display(HTML(f"""
        <div style="display: flex; justify-content: space-around; background: #f8f9fa; border: 1px solid #dee2e6; color: #2b2d42; padding: 20px; border-radius: 12px; font-family: sans-serif; margin-bottom: 25px; box-shadow: 2px 2px 10px rgba(0,0,0,0.05);">
            <div style="text-align:center; border-right: 1px solid #dee2e6; padding-right: 20px;">
                <p style="font-size:12px; margin:0; color: #6c757d; font-weight: bold; text-transform: uppercase;">Ingreso Neto</p>
                <h2 style="margin:5px 0; color: #4472C4;">${ing_neto:,.2f}</h2>
            </div>
            <div style="text-align:center; border-right: 1px solid #dee2e6; padding: 0 20px;">
                <p style="font-size:12px; margin:0; color: #6c757d; font-weight: bold; text-transform: uppercase;">Margen Estimado</p>
                <h2 style="margin:5px 0; color: #70AD47;">${marg_est:,.2f} <span style="font-size:16px;">({marg_pct:.1%})</span></h2>
            </div>
            <div style="text-align:center; padding-left: 20px;">
                <p style="font-size:12px; margin:0; color: #6c757d; font-weight: bold; text-transform: uppercase;">Entrega a Tiempo</p>
                <h2 style="margin:5px 0; color: #FFC000;">{tasa_ent:.1%}</h2>
            </div>
        </div>
    """))

    # --- GENERAR GRÁFICAS ---

    # G1: Ingreso vs Margen por Familia
    res_fam = df_f.groupby('Familia').agg({'Ingreso_Neto':'sum', 'Margen_Estimado':'sum'}).reset_index()
    fig1 = px.bar(res_fam, x='Familia', y=['Ingreso_Neto', 'Margen_Estimado'],
                 barmode='group', title="<b>📊 Análisis Comercial: Ingreso vs Margen por Familia</b>",
                 color_discrete_sequence=['#4472C4', '#70AD47'], template='plotly_white')
    fig1.show()

    # G2: Descuento % por Canal
    res_can = df_f.groupby('Canal').agg({'Descuento_Porc':'mean'}).reset_index()
    fig2 = px.line(res_can, x='Canal', y='Descuento_Porc',
                  title="<b>📉 Presión Comercial: Descuento Promedio por Canal</b>",
                  markers=True, template='plotly_white')
    fig2.update_yaxes(tickformat=".1%")
    fig2.show()

    # G3: Merma Operativa (si hay datos)
    if not dp_f.empty:
        res_op = dp_f.groupby('Linea').agg({'Merma_Pct':'mean'}).reset_index()
        fig3 = px.bar(res_op, x='Linea', y='Merma_Pct',
                     title="<b>🏭 Eficiencia Operativa: % Merma por Línea</b>",
                     color='Merma_Pct', color_continuous_scale='RdYlGn_r')
        fig3.add_hline(y=0.0405, line_dash="dash", line_color="red", annotation_text="Meta (4.05%)")
        fig3.update_yaxes(tickformat=".2%")
        fig3.show()
    else:
        display(HTML("<p style='color:orange;'>ℹ️ No hay datos de producción para las regiones seleccionadas.</p>"))

In [42]:
# ============================================
# TABLAS DINÁMICAS - ANÁLISIS OPERATIVO
# ============================================

import pandas as pd
import numpy as np
from IPython.display import display, HTML

# Asegurar que existe la columna Turno (simulada si no existe)
if 'Turno' not in df_produccion.columns:
    np.random.seed(42)
    df_produccion['Turno'] = np.random.choice(['Matutino', 'Vespertino', 'Nocturno'], size=len(df_produccion))

display(HTML("<h2>🏭 TABLAS DINÁMICAS OPERATIVAS</h2>"))

# TABLA 1: Producción por Planta y Línea
pivot_produccion = pd.pivot_table(
    df_produccion,
    values=['Unidades_Producidas', 'Unidades_Merma', 'Tiempo_Muerto_Min', 'Costo_Total_Lote'],
    index=['Planta', 'Linea'],
    aggfunc={
        'Unidades_Producidas': 'sum',
        'Unidades_Merma': 'sum',
        'Tiempo_Muerto_Min': 'sum',
        'Costo_Total_Lote': 'sum'
    }
)
pivot_produccion['% Merma'] = pivot_produccion['Unidades_Merma'] / pivot_produccion['Unidades_Producidas']
pivot_produccion['Costo_por_Unidad'] = pivot_produccion['Costo_Total_Lote'] / pivot_produccion['Unidades_Producidas']

display(HTML("<h3>📊 Tabla 1: Producción por Planta y Línea</h3>"))
display(pivot_produccion.style.format({
    'Unidades_Producidas': '{:,.0f}',
    'Unidades_Merma': '{:,.0f}',
    'Tiempo_Muerto_Min': '{:,.0f} min',
    'Costo_Total_Lote': '${:,.2f}',
    '% Merma': '{:.2%}',
    'Costo_por_Unidad': '${:,.2f}'
}))

# TABLA 2: Eficiencia por Turno
pivot_turno = df_produccion.groupby(['Planta', 'Turno']).agg({
    'Unidades_Producidas': 'sum',
    'Unidades_Merma': 'sum',
    'Tiempo_Muerto_Min': 'sum'
})
pivot_turno['% Merma'] = pivot_turno['Unidades_Merma'] / pivot_turno['Unidades_Producidas']

display(HTML("<h3>📊 Tabla 2: Eficiencia por Planta y Turno</h3>"))
display(pivot_turno.style.format({
    'Unidades_Producidas': '{:,.0f}',
    'Unidades_Merma': '{:,.0f}',
    'Tiempo_Muerto_Min': '{:,.0f} min',
    '% Merma': '{:.2%}'
}))

# TABLA 3: Resumen de Merma por Línea (ordenada de peor a mejor)
merma_linea = df_produccion.groupby('Linea').agg({
    'Unidades_Producidas': 'sum',
    'Unidades_Merma': 'sum',
    'Tiempo_Muerto_Min': 'sum'
}).round(2)
merma_linea['% Merma'] = (merma_linea['Unidades_Merma'] / merma_linea['Unidades_Producidas']).round(4)
merma_linea = merma_linea.sort_values('% Merma', ascending=False)

display(HTML("<h3>📊 Tabla 3: Ranking de Merma por Línea (Peor a Mejor)</h3>"))
display(merma_linea.style.format({
    'Unidades_Producidas': '{:,.0f}',
    'Unidades_Merma': '{:,.0f}',
    'Tiempo_Muerto_Min': '{:,.0f} min',
    '% Merma': '{:.2%}'
}).bar(subset=['% Merma'], color='#FF6B6B'))

print("✅ Tablas dinámicas operativas cargadas correctamente.")

,Unidades_Producidas,Unidades_Merma,Tiempo_Muerto_Min,% Merma
Linea,,,,
L4 Cápsulas,"121,188","7,291","4,925 min",6.02%
L3 Empaque,"133,292","7,513","5,192 min",5.64%
L1 Tostado,"131,730","4,935","2,905 min",3.75%
L2 Molido,"186,445","6,116","4,259 min",3.28%


✅ Tablas dinámicas operativas cargadas correctamente.


In [43]:
# ============================================
# SEGMENTADORES - DASHBOARD OPERATIVO
# ============================================

import ipywidgets as widgets
from IPython.display import display

display(HTML("<h2>🎛️ SEGMENTADORES OPERATIVOS</h2>"))
display(HTML("<p>Selecciona los valores para filtrar el Dashboard Operativo:</p>"))

# Segmentador 1: Planta
slicer_planta = widgets.SelectMultiple(
    options=sorted(df_produccion['Planta'].unique()),
    value=list(df_produccion['Planta'].unique()),
    description='🏭 Planta:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='30%')
)

# Segmentador 2: Línea
slicer_linea = widgets.SelectMultiple(
    options=sorted(df_produccion['Linea'].unique()),
    value=list(df_produccion['Linea'].unique()),
    description='⚙️ Línea:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='30%')
)

# Segmentador 3: Turno
slicer_turno = widgets.SelectMultiple(
    options=sorted(df_produccion['Turno'].unique()),
    value=list(df_produccion['Turno'].unique()),
    description='🕒 Turno:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='30%')
)

# Botón para aplicar filtros
btn_aplicar_op = widgets.Button(
    description="🔍 APLICAR FILTROS OPERATIVOS",
    button_style='success',
    layout=widgets.Layout(width='50%', margin='10px 0px')
)

# Contenedor horizontal de los slicers
slicers_row = widgets.HBox([slicer_planta, slicer_linea, slicer_turno])

display(slicers_row)
display(btn_aplicar_op)

# Variable global para guardar los filtros seleccionados
filtros_op_seleccionados = {
    'planta': list(df_produccion['Planta'].unique()),
    'linea': list(df_produccion['Linea'].unique()),
    'turno': list(df_produccion['Turno'].unique())
}

def aplicar_filtros_op(b):
    global filtros_op_seleccionados
    filtros_op_seleccionados['planta'] = list(slicer_planta.value) if slicer_planta.value else list(df_produccion['Planta'].unique())
    filtros_op_seleccionados['linea'] = list(slicer_linea.value) if slicer_linea.value else list(df_produccion['Linea'].unique())
    filtros_op_seleccionados['turno'] = list(slicer_turno.value) if slicer_turno.value else list(df_produccion['Turno'].unique())
    print(f"✅ Filtros operativos actualizados:")
    print(f"   - Plantas: {filtros_op_seleccionados['planta']}")
    print(f"   - Líneas: {filtros_op_seleccionados['linea']}")
    print(f"   - Turnos: {filtros_op_seleccionados['turno']}")
    print("👉 Ahora ejecuta la CELDA 7 (KPIs y Gráficas Operativas) para ver los cambios.")

btn_aplicar_op.on_click(aplicar_filtros_op)

print("✅ Segmentadores operativos listos. Haz clic en 'APLICAR FILTROS' después de seleccionar.")

Button(button_style='success', description='🔍 APLICAR FILTROS OPERATIVOS', layout=Layout(margin='10px 0px', wi…

✅ Segmentadores operativos listos. Haz clic en 'APLICAR FILTROS' después de seleccionar.
✅ Filtros operativos actualizados:
   - Plantas: ['Planta Norte', 'Planta Occidente']
   - Líneas: ['L1 Tostado', 'L4 Cápsulas']
   - Turnos: ['Matutino', 'Vespertino']
👉 Ahora ejecuta la CELDA 7 (KPIs y Gráficas Operativas) para ver los cambios.


In [45]:
# ============================================
# DASHBOARD OPERATIVO - KPIs y GRÁFICAS
# ============================================

from IPython.display import clear_output
import plotly.express as px
import plotly.graph_objects as go

# Aplicar filtros seleccionados en la Celda 6
df_op_filtrado = df_produccion[
    (df_produccion['Planta'].isin(filtros_op_seleccionados['planta'])) &
    (df_produccion['Linea'].isin(filtros_op_seleccionados['linea'])) &
    (df_produccion['Turno'].isin(filtros_op_seleccionados['turno']))
]

display(HTML("<h2>🏭 DASHBOARD OPERATIVO</h2>"))

if df_op_filtrado.empty:
    display(HTML("<p style='color:red;'>⚠️ No hay datos con los filtros seleccionados. Ajusta los segmentadores.</p>"))
else:
    # ========== KPIs ==========
    total_unidades = df_op_filtrado['Unidades_Producidas'].sum()
    total_merma = df_op_filtrado['Unidades_Merma'].sum()
    pct_merma = (total_merma / total_unidades) if total_unidades > 0 else 0
    total_tiempo_muerto = df_op_filtrado['Tiempo_Muerto_Min'].sum()
    costo_total = df_op_filtrado['Costo_Total_Lote'].sum()
    costo_por_unidad_buena = costo_total / (total_unidades - total_merma) if (total_unidades - total_merma) > 0 else 0

    # Mostrar KPIs en tarjetas
    display(HTML(f"""
        <div style="display: flex; justify-content: space-around; background: #1a1a2e; color: white; padding: 25px; border-radius: 15px; margin-bottom: 30px;">
            <div style="text-align: center;">
                <p style="font-size: 12px; margin:0; opacity:0.7;">📦 UNIDADES PRODUCIDAS</p>
                <h2 style="margin:5px 0; color: #4CC9F0;">{total_unidades:,.0f}</h2>
            </div>
            <div style="text-align: center;">
                <p style="font-size: 12px; margin:0; opacity:0.7;">⚠️ % MERMA</p>
                <h2 style="margin:5px 0; color: #F72585;">{pct_merma:.2%}</h2>
            </div>
            <div style="text-align: center;">
                <p style="font-size: 12px; margin:0; opacity:0.7;">⏱️ TIEMPO MUERTO</p>
                <h2 style="margin:5px 0; color: #F8961E;">{total_tiempo_muerto:,.0f} min</h2>
            </div>
            <div style="text-align: center;">
                <p style="font-size: 12px; margin:0; opacity:0.7;">💰 COSTO / UNIDAD BUENA</p>
                <h2 style="margin:5px 0; color: #4CAF50;">${costo_por_unidad_buena:.2f}</h2>
            </div>
        </div>
    """))

    # ========== GRÁFICA 1: Merma por Línea ==========
    merma_por_linea = df_op_filtrado.groupby('Linea')['Merma_Pct'].mean().reset_index()
    fig1 = px.bar(merma_por_linea, x='Linea', y='Merma_Pct',
                  title="<b>📊 % MERMA PROMEDIO POR LÍNEA</b>",
                  color='Merma_Pct', color_continuous_scale='RdYlGn_r',
                  text_auto='.2%')
    fig1.add_hline(y=0.045, line_dash="dash", line_color="red",
                   annotation_text="Meta (4.5%)")
    fig1.update_layout(height=450, template='plotly_white')
    fig1.show()

    # ========== GRÁFICA 2: Tiempo Muerto por Planta ==========
    tiempo_por_planta = df_op_filtrado.groupby('Planta')['Tiempo_Muerto_Min'].sum().reset_index()
    fig2 = px.bar(tiempo_por_planta, x='Planta', y='Tiempo_Muerto_Min',
                  title="<b>⏱️ TIEMPO MUERTO TOTAL POR PLANTA</b>",
                  color='Tiempo_Muerto_Min', color_continuous_scale='Oranges',
                  text_auto=True)
    fig2.update_layout(height=450, template='plotly_white')
    fig2.show()

    # ========== GRÁFICA 3: Comparativa por Turno ==========
    turno_metrics = df_op_filtrado.groupby('Turno').agg({
        'Merma_Pct': 'mean',
        'Tiempo_Muerto_Min': 'sum'
    }).reset_index()

    fig3 = go.Figure()
    fig3.add_trace(go.Bar(x=turno_metrics['Turno'], y=turno_metrics['Merma_Pct'],
                          name='% Merma', marker_color='#F72585',
                          text=turno_metrics['Merma_Pct'].apply(lambda x: f'{x:.2%}'),
                          textposition='outside'))
    fig3.add_trace(go.Bar(x=turno_metrics['Turno'], y=turno_metrics['Tiempo_Muerto_Min'] / 60,
                          name='Tiempo Muerto (horas)', marker_color='#F8961E',
                          text=turno_metrics['Tiempo_Muerto_Min'].apply(lambda x: f'{x/60:.1f}h'),
                          textposition='outside'))
    fig3.update_layout(title="<b>📊 COMPARATIVA POR TURNO: MERMA vs TIEMPO MUERTO</b>",
                       barmode='group', height=450, template='plotly_white')
    fig3.show()

    print(f"\n✅ Dashboard actualizado con {len(df_op_filtrado)} registros operativos.")


✅ Dashboard actualizado con 57 registros operativos.


In [46]:
# ============================================
# TABLAS DINÁMICAS - ANÁLISIS EJECUTIVO
# ============================================

from IPython.display import display, HTML

display(HTML("<h2>👔 TABLAS DINÁMICAS EJECUTIVAS</h2>"))

# TABLA 1: Rentabilidad por Región y Canal
pivot_rentabilidad = pd.pivot_table(
    df_maestra,
    values=['Ingreso_Neto', 'Margen_Estimado'],
    index=['Region'],
    columns=['Canal'],
    aggfunc='sum',
    fill_value=0
)

# Calcular margen %
for canal in pivot_rentabilidad['Margen_Estimado'].columns:
    pivot_rentabilidad[('Margen_%', canal)] = (pivot_rentabilidad['Margen_Estimado'][canal] /
                                                pivot_rentabilidad['Ingreso_Neto'][canal])

display(HTML("<h3>📊 Tabla 1: Rentabilidad por Región y Canal</h3>"))
display(pivot_rentabilidad.style.format({
    ('Ingreso_Neto', ''): '${:,.2f}',
    ('Margen_Estimado', ''): '${:,.2f}',
    ('Margen_%', ''): '{:.2%}'
}))

# TABLA 2: TOP 5 Productos (Familias) por Margen
top_familias = df_maestra.groupby('Familia').agg({
    'Ingreso_Neto': 'sum',
    'Margen_Estimado': 'sum',
    'Unidades': 'sum'
}).round(2)
top_familias['Margen_%'] = top_familias['Margen_Estimado'] / top_familias['Ingreso_Neto']
top_familias = top_familias.sort_values('Margen_Estimado', ascending=False)

display(HTML("<h3>📊 Tabla 2: Rentabilidad por Familia de Producto</h3>"))
display(top_familias.style.format({
    'Ingreso_Neto': '${:,.2f}',
    'Margen_Estimado': '${:,.2f}',
    'Unidades': '{:,.0f}',
    'Margen_%': '{:.2%}'
}).bar(subset=['Margen_Estimado'], color='#70AD47'))

# TABLA 3: Cumplimiento vs Presupuesto
cumplimiento = df_maestra.groupby(['Region', 'Canal']).agg({
    'Ingreso_Neto': 'sum',
    'Presupuesto_Ventas': 'sum'
}).round(2)
cumplimiento['Cumplimiento_%'] = cumplimiento['Ingreso_Neto'] / cumplimiento['Presupuesto_Ventas']
cumplimiento['Variación_$'] = cumplimiento['Ingreso_Neto'] - cumplimiento['Presupuesto_Ventas']

display(HTML("<h3>📊 Tabla 3: Cumplimiento de Ventas vs Presupuesto</h3>"))
display(cumplimiento.style.format({
    'Ingreso_Neto': '${:,.2f}',
    'Presupuesto_Ventas': '${:,.2f}',
    'Cumplimiento_%': '{:.2%}',
    'Variación_$': '${:,.2f}'
}).background_gradient(subset=['Cumplimiento_%'], cmap='RdYlGn'))

print("✅ Tablas dinámicas ejecutivas cargadas correctamente.")

,Ingreso_Neto,Margen_Estimado,Unidades,Margen_%
Familia,,,,
Especialidad,"$20,671,943.79","$5,495,260.57","77,034",26.58%
Tradicional,"$15,937,767.04","$4,298,946.71","112,630",26.97%
Cápsulas,"$7,978,514.39","$3,181,281.15","88,557",39.87%
Premium,"$10,743,749.40","$2,967,060.18","47,525",27.62%
Orgánico,"$11,269,210.68","$2,859,064.24","45,065",25.37%
Descafeinado,"$5,273,981.14","$1,396,930.62","32,586",26.49%
Bebidas listas,"$2,143,388.35","$694,957.35","14,220",32.42%


✅ Tablas dinámicas ejecutivas cargadas correctamente.


In [47]:
# ============================================
# SEGMENTADORES - DASHBOARD EJECUTIVO
# ============================================

import ipywidgets as widgets
from IPython.display import display

display(HTML("<h2>🎛️ SEGMENTADORES EJECUTIVOS</h2>"))
display(HTML("<p>Selecciona los valores para filtrar el Dashboard Ejecutivo:</p>"))

# Segmentador 1: Región
slicer_region_ejec = widgets.SelectMultiple(
    options=sorted(df_maestra['Region'].unique()),
    value=list(df_maestra['Region'].unique()),
    description='📍 Región:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='32%')
)

# Segmentador 2: Canal
slicer_canal_ejec = widgets.SelectMultiple(
    options=sorted(df_maestra['Canal'].unique()),
    value=list(df_maestra['Canal'].unique()),
    description='🛒 Canal:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='32%')
)

# Segmentador 3: Segmento de Cliente
slicer_segmento = widgets.SelectMultiple(
    options=sorted(df_maestra['Segmento_Cliente'].dropna().unique()),
    value=list(df_maestra['Segmento_Cliente'].dropna().unique()),
    description='🏢 Segmento:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='32%')
)

# Botón para aplicar filtros
btn_aplicar_ejec = widgets.Button(
    description="🔍 APLICAR FILTROS EJECUTIVOS",
    button_style='success',
    layout=widgets.Layout(width='50%', margin='10px 0px')
)

# Contenedor horizontal
slicers_row_ejec = widgets.HBox([slicer_region_ejec, slicer_canal_ejec, slicer_segmento])

display(slicers_row_ejec)
display(btn_aplicar_ejec)

# Variable global para guardar los filtros seleccionados
filtros_ejec_seleccionados = {
    'region': list(df_maestra['Region'].unique()),
    'canal': list(df_maestra['Canal'].unique()),
    'segmento': list(df_maestra['Segmento_Cliente'].dropna().unique())
}

def aplicar_filtros_ejec(b):
    global filtros_ejec_seleccionados
    filtros_ejec_seleccionados['region'] = list(slicer_region_ejec.value) if slicer_region_ejec.value else list(df_maestra['Region'].unique())
    filtros_ejec_seleccionados['canal'] = list(slicer_canal_ejec.value) if slicer_canal_ejec.value else list(df_maestra['Canal'].unique())
    filtros_ejec_seleccionados['segmento'] = list(slicer_segmento.value) if slicer_segmento.value else list(df_maestra['Segmento_Cliente'].dropna().unique())
    print(f"✅ Filtros ejecutivos actualizados:")
    print(f"   - Regiones: {filtros_ejec_seleccionados['region']}")
    print(f"   - Canales: {filtros_ejec_seleccionados['canal']}")
    print(f"   - Segmentos: {filtros_ejec_seleccionados['segmento']}")
    print("👉 Ahora ejecuta la CELDA 10 (KPIs y Gráficas Ejecutivas) para ver los cambios.")

btn_aplicar_ejec.on_click(aplicar_filtros_ejec)

print("✅ Segmentadores ejecutivos listos. Haz clic en 'APLICAR FILTROS' después de seleccionar.")

Button(button_style='success', description='🔍 APLICAR FILTROS EJECUTIVOS', layout=Layout(margin='10px 0px', wi…

✅ Segmentadores ejecutivos listos. Haz clic en 'APLICAR FILTROS' después de seleccionar.
✅ Filtros ejecutivos actualizados:
   - Regiones: ['Norte']
   - Canales: ['Autoservicio', 'Mayoreo']
   - Segmentos: ['Cadena nacional', 'Cafetería premium', 'Marketplace', 'Minorista independiente']
👉 Ahora ejecuta la CELDA 10 (KPIs y Gráficas Ejecutivas) para ver los cambios.
✅ Filtros ejecutivos actualizados:
   - Regiones: ['Centro', 'Norte', 'Occidente']
   - Canales: ['Autoservicio', 'E-commerce', 'HORECA', 'Mayoreo']
   - Segmentos: ['Cadena nacional', 'Cafetería premium', 'Corporativo', 'Distribuidor regional', 'Marketplace', 'Minorista independiente']
👉 Ahora ejecuta la CELDA 10 (KPIs y Gráficas Ejecutivas) para ver los cambios.


In [53]:
# ============================================
# DASHBOARD EJECUTIVO - KPIs y GRÁFICAS
# ============================================

import plotly.express as px
import plotly.graph_objects as go

# Aplicar filtros seleccionados en la Celda 9
df_ejec_filtrado = df_maestra[
    (df_maestra['Region'].isin(filtros_ejec_seleccionados['region'])) &
    (df_maestra['Canal'].isin(filtros_ejec_seleccionados['canal'])) &
    (df_maestra['Segmento_Cliente'].isin(filtros_ejec_seleccionados['segmento']))
]

display(HTML("<h2>👔 DASHBOARD EJECUTIVO INTEGRADOR</h2>"))

if df_ejec_filtrado.empty:
    display(HTML("<p style='color:red;'>⚠️ No hay datos con los filtros seleccionados. Ajusta los segmentadores.</p>"))
else:
    # ========== KPIs PRINCIPALES ==========
    ventas_netas = df_ejec_filtrado['Ingreso_Neto'].sum()
    margen_total = df_ejec_filtrado['Margen_Estimado'].sum()
    margen_pct = (margen_total / ventas_netas) if ventas_netas > 0 else 0
    descuento_prom = df_ejec_filtrado['Descuento_Porc'].mean()
    devolucion_pct = (df_ejec_filtrado['Unidades_Devueltas'].sum() / df_ejec_filtrado['Unidades'].sum()) if df_ejec_filtrado['Unidades'].sum() > 0 else 0

    # Calcular cumplimiento vs presupuesto
    presupuesto_total = df_ejec_filtrado['Presupuesto_Ventas'].sum()
    cumplimiento_pct = (ventas_netas / presupuesto_total) if presupuesto_total > 0 else 0

    # Mostrar KPIs estilo tablero
    display(HTML(f"""
        <div style="display: flex; justify-content: space-around; background: linear-gradient(135deg, #1a1a2e 0%, #16213e 100%); color: white; padding: 25px; border-radius: 15px; margin-bottom: 30px; flex-wrap: wrap;">
            <div style="text-align: center; padding: 10px;">
                <p style="font-size: 12px; margin:0; opacity:0.8;">💰 VENTAS NETAS</p>
                <h2 style="margin:5px 0; color: #4CC9F0;">${ventas_netas:,.2f}</h2>
            </div>
            <div style="text-align: center; padding: 10px;">
                <p style="font-size: 12px; margin:0; opacity:0.8;">📈 MARGEN TOTAL</p>
                <h2 style="margin:5px 0; color: #4CAF50;">${margen_total:,.2f} <span style="font-size:16px;">({margen_pct:.1%})</span></h2>
            </div>
            <div style="text-align: center; padding: 10px;">
                <p style="font-size: 12px; margin:0; opacity:0.8;">🎯 CUMPLIMIENTO</p>
                <h2 style="margin:5px 0; color: #F8961E;">{cumplimiento_pct:.1%}</h2>
            </div>
            <div style="text-align: center; padding: 10px;">
                <p style="font-size: 12px; margin:0; opacity:0.8;">🏷️ DESCUENTO PROM.</p>
                <h2 style="margin:5px 0; color: #F72585;">{descuento_prom:.1%}</h2>
            </div>
            <div style="text-align: center; padding: 10px;">
                <p style="font-size: 12px; margin:0; opacity:0.8;">📦 DEVOLUCIÓN</p>
                <h2 style="margin:5px 0; color: #FFB703;">{devolucion_pct:.1%}</h2>
            </div>
        </div>
    """))

    # ========== GRÁFICA 1: Ventas vs Margen por Familia ==========
    resumen_familias = df_ejec_filtrado.groupby('Familia').agg({
        'Ingreso_Neto': 'sum',
        'Margen_Estimado': 'sum'
    }).reset_index()

    fig_ejec1 = px.bar(resumen_familias, x='Familia', y=['Ingreso_Neto', 'Margen_Estimado'],
                       title="<b>📊 VENTAS vs MARGEN POR FAMILIA</b>",
                       barmode='group',
                       color_discrete_sequence=['#4472C4', '#70AD47'],
                       template='plotly_white',
                       labels={'value': 'Monto ($)', 'variable': 'Métrica', 'Familia': 'Familia de Producto'})
    fig_ejec1.update_layout(height=450, legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1))
    fig_ejec1.update_yaxes(tickprefix='$')  # CORREGIDO: update_yaxes con 'e'
    fig_ejec1.show()

    # ========== GRÁFICA 2: Margen por Canal (Gráfico de barras horizontal) ==========
    margen_canal = df_ejec_filtrado.groupby('Canal')['Margen_Pct'].mean().reset_index().sort_values('Margen_Pct', ascending=True)

    fig_ejec2 = px.bar(margen_canal, x='Margen_Pct', y='Canal',
                       title="<b>🎯 RENTABILIDAD POR CANAL (% Margen)</b>",
                       color='Margen_Pct',
                       color_continuous_scale='RdYlGn',
                       orientation='h',
                       text_auto='.1%',
                       template='plotly_white',
                       labels={'Margen_Pct': '% Margen', 'Canal': 'Canal de Venta'})
    fig_ejec2.update_layout(height=400, coloraxis_colorbar=dict(title="Margen %"))
    fig_ejec2.update_xaxes(tickformat='.0%')  # CORREGIDO: update_xaxes con 'e'
    fig_ejec2.show()

    # ========== GRÁFICA 3: Matriz de calor - Rentabilidad por Región y Canal ==========
    matriz_rentabilidad = pd.pivot_table(
        df_ejec_filtrado,
        values='Margen_Pct',
        index='Region',
        columns='Canal',
        aggfunc='mean',
        fill_value=0
    )

    fig_ejec3 = px.imshow(matriz_rentabilidad,
                          title="<b>🔥 MATRIZ DE RENTABILIDAD: Región vs Canal</b>",
                          text_auto='.1%',
                          color_continuous_scale='RdYlGn',
                          aspect='auto',
                          template='plotly_white',
                          labels={'color': 'Margen %', 'x': 'Canal', 'y': 'Región'})
    fig_ejec3.update_layout(height=450)
    fig_ejec3.show()

    # ========== GRÁFICA 4: Cumplimiento de presupuesto por Región ==========
    cumplimiento_region = df_ejec_filtrado.groupby('Region').agg({
        'Ingreso_Neto': 'sum',
        'Presupuesto_Ventas': 'sum'
    }).reset_index()
    cumplimiento_region['Cumplimiento_%'] = (cumplimiento_region['Ingreso_Neto'] / cumplimiento_region['Presupuesto_Ventas']) * 100
    cumplimiento_region = cumplimiento_region.sort_values('Cumplimiento_%', ascending=False)

    fig_ejec4 = px.bar(cumplimiento_region, x='Region', y='Cumplimiento_%',
                       title="<b>🎯 CUMPLIMIENTO DE PRESUPUESTO POR REGIÓN</b>",
                       color='Cumplimiento_%',
                       color_continuous_scale='RdYlGn',
                       text_auto='.1f',
                       template='plotly_white',
                       labels={'Cumplimiento_%': 'Cumplimiento (%)', 'Region': 'Región'})
    fig_ejec4.add_hline(y=100, line_dash="dash", line_color="red", annotation_text="Meta (100%)")
    fig_ejec4.update_layout(height=450)
    fig_ejec4.update_yaxes(title='% Cumplimiento', range=[0, max(cumplimiento_region['Cumplimiento_%'].max() + 20, 120)])  # CORREGIDO
    fig_ejec4.show()

    # ========== INSIGHTS ESTRATÉGICOS ==========
    # Identificar mejores y peores
    mejor_familia = resumen_familias.loc[resumen_familias['Margen_Estimado'].idxmax(), 'Familia']
    peor_familia = resumen_familias.loc[resumen_familias['Margen_Estimado'].idxmin(), 'Familia']
    mejor_canal = margen_canal.iloc[-1]['Canal'] if len(margen_canal) > 0 else 'N/A'
    peor_canal = margen_canal.iloc[0]['Canal'] if len(margen_canal) > 0 else 'N/A'
    mejor_region = cumplimiento_region.iloc[0]['Region'] if len(cumplimiento_region) > 0 else 'N/A'
    peor_region = cumplimiento_region.iloc[-1]['Region'] if len(cumplimiento_region) > 0 else 'N/A'

    # Determinar si el margen está en target (suponiendo target 28%)
    target_margen = 0.28
    status_margen = "✅ SUPERIOR al objetivo" if margen_pct >= target_margen else "⚠️ INFERIOR al objetivo"
    color_status = "#4CAF50" if margen_pct >= target_margen else "#F72585"

    display(HTML(f"""
        <div style="background: #1a1a2e; padding: 25px; border-radius: 12px; margin-top: 20px; color: white;">
            <h3 style="margin-top: 0;">📌 CONCLUSIONES EJECUTIVAS</h3>
            <div style="display: flex; flex-wrap: wrap; gap: 20px; justify-content: space-between;">
                <div style="flex: 1; min-width: 200px; background: #16213e; padding: 15px; border-radius: 10px;">
                    <h4 style="margin: 0 0 10px 0; color: #4CC9F0;">✅ OPORTUNIDADES</h4>
                    <ul style="margin: 0; padding-left: 20px;">
                        <li><strong>{mejor_familia}</strong> - Familia más rentable</li>
                        <li><strong>{mejor_canal}</strong> - Canal con mejor margen</li>
                        <li><strong>{mejor_region}</strong> - Mejor cumplimiento de presupuesto</li>
                    </ul>
                </div>
                <div style="flex: 1; min-width: 200px; background: #16213e; padding: 15px; border-radius: 10px;">
                    <h4 style="margin: 0 0 10px 0; color: #F72585;">⚠️ FOCO ROJO</h4>
                    <ul style="margin: 0; padding-left: 20px;">
                        <li><strong>{peor_familia}</strong> - Familia menos rentable</li>
                        <li><strong>{peor_canal}</strong> - Canal con menor margen</li>
                        <li><strong>{peor_region}</strong> - Peor cumplimiento de presupuesto</li>
                    </ul>
                </div>
                <div style="flex: 1; min-width: 200px; background: #16213e; padding: 15px; border-radius: 10px;">
                    <h4 style="margin: 0 0 10px 0; color: #F8961E;">📊 MÉTRICAS CLAVE</h4>
                    <ul style="margin: 0; padding-left: 20px;">
                        <li>Margen Global: <strong style="color: {color_status};">{margen_pct:.1%} {status_margen}</strong></li>
                        <li>Cumplimiento Presupuesto: <strong>{cumplimiento_pct:.1%}</strong></li>
                        <li>Descuento Promedio: <strong>{descuento_prom:.1%}</strong></li>
                    </ul>
                </div>
            </div>
            <hr style="margin: 20px 0; border-color: #333;">
            <h4>🎯 3 DECISIONES RECOMENDADAS PARA EL SIGUIENTE TRIMESTRE:</h4>
            <ol style="font-size: 14px; line-height: 1.8;">
                <li><strong>Invertir en {mejor_familia}:</strong> Aumentar producción y promoción de la familia más rentable.</li>
                <li><strong>Revisar estrategia en {peor_canal}:</strong> Evaluar política de descuentos y optimizar costos de distribución.</li>
                <li><strong>Enfocar esfuerzos en {mejor_region}:</strong> Duplicar inversión de marketing en la región con mejor desempeño.</li>
            </ol>
        </div>
    """))

    print(f"\n✅ Dashboard ejecutivo actualizado con {len(df_ejec_filtrado):,} registros comerciales.")
    print(f"   - Período analizado: {df_ejec_filtrado['Fecha_Pedido'].min().strftime('%b %Y')} a {df_ejec_filtrado['Fecha_Pedido'].max().strftime('%b %Y')}")


✅ Dashboard ejecutivo actualizado con 4,000 registros comerciales.
   - Período analizado: Sep 2025 a Dec 2025


In [54]:
# ============================================
# MACRO: GENERAR REPORTE EJECUTIVO COMPLETO
# ============================================

from IPython.display import display, HTML, clear_output
import plotly.io as pio
from datetime import datetime
import io
import base64

# Datos del equipo
miembros_equipo = [
    "Almanza Frausto Eduardo Jose Francisco",
    "Juan Diego Argüello Nájera",
    "Hernandez Hernandez Abilene",
    "Salgado Quintero Ana Daniela",
    "Velazquez Aguilar Oscar"
]

profesor = "Jorge Tovar García"
materia = "Informática Aplicada"
grupo = "26-I"

def generar_reporte_ejecutivo(b):
    """Macro que genera un reporte ejecutivo con todos los resultados del Dashboard"""

    clear_output(wait=True)

    print("=" * 70)
    print("📄 MACRO: GENERANDO REPORTE EJECUTIVO")
    print("=" * 70)

    # Re-aplicar filtros actuales para asegurar datos frescos
    df_reporte = df_maestra[
        (df_maestra['Region'].isin(filtros_ejec_seleccionados['region'])) &
        (df_maestra['Canal'].isin(filtros_ejec_seleccionados['canal'])) &
        (df_maestra['Segmento_Cliente'].isin(filtros_ejec_seleccionados['segmento']))
    ]

    if df_reporte.empty:
        print("⚠️ No hay datos con los filtros seleccionados. No se puede generar el reporte.")
        return

    # Calcular KPIs
    ventas_netas = df_reporte['Ingreso_Neto'].sum()
    margen_total = df_reporte['Margen_Estimado'].sum()
    margen_pct = (margen_total / ventas_netas) if ventas_netas > 0 else 0
    descuento_prom = df_reporte['Descuento_Porc'].mean()
    devolucion_pct = (df_reporte['Unidades_Devueltas'].sum() / df_reporte['Unidades'].sum()) if df_reporte['Unidades'].sum() > 0 else 0
    presupuesto_total = df_reporte['Presupuesto_Ventas'].sum()
    cumplimiento_pct = (ventas_netas / presupuesto_total) if presupuesto_total > 0 else 0

    # Identificar mejores y peores
    resumen_familias = df_reporte.groupby('Familia').agg({'Ingreso_Neto': 'sum', 'Margen_Estimado': 'sum'}).reset_index()
    margen_canal = df_reporte.groupby('Canal')['Margen_Pct'].mean().reset_index().sort_values('Margen_Pct', ascending=True)
    cumplimiento_region = df_reporte.groupby('Region').agg({'Ingreso_Neto': 'sum', 'Presupuesto_Ventas': 'sum'}).reset_index()
    cumplimiento_region['Cumplimiento_%'] = (cumplimiento_region['Ingreso_Neto'] / cumplimiento_region['Presupuesto_Ventas']) * 100

    mejor_familia = resumen_familias.loc[resumen_familias['Margen_Estimado'].idxmax(), 'Familia'] if not resumen_familias.empty else 'N/A'
    peor_familia = resumen_familias.loc[resumen_familias['Margen_Estimado'].idxmin(), 'Familia'] if not resumen_familias.empty else 'N/A'
    mejor_canal = margen_canal.iloc[-1]['Canal'] if len(margen_canal) > 0 else 'N/A'
    peor_canal = margen_canal.iloc[0]['Canal'] if len(margen_canal) > 0 else 'N/A'
    mejor_region = cumplimiento_region.iloc[0]['Region'] if len(cumplimiento_region) > 0 else 'N/A'

    # Fecha actual
    fecha_generacion = datetime.now().strftime("%d de %B de %Y - %H:%M hrs")

    # ========== GENERAR HTML DEL REPORTE ==========

    # Tabla de miembros del equipo
    miembros_html = "".join([f"<li>{m}</li>" for m in miembros_equipo])

    html_reporte = f"""
    <!DOCTYPE html>
    <html lang="es">
    <head>
        <meta charset="UTF-8">
        <meta name="viewport" content="width=device-width, initial-scale=1.0">
        <title>Reporte Ejecutivo - Café Monteloma</title>
        <style>
            * {{
                margin: 0;
                padding: 0;
                box-sizing: border-box;
            }}
            body {{
                font-family: 'Segoe UI', Arial, sans-serif;
                background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
                padding: 40px 20px;
            }}
            .report-container {{
                max-width: 1200px;
                margin: 0 auto;
                background: white;
                border-radius: 20px;
                box-shadow: 0 20px 60px rgba(0,0,0,0.3);
                overflow: hidden;
            }}
            /* HEADER */
            .header {{
                background: linear-gradient(135deg, #1a1a2e 0%, #16213e 100%);
                color: white;
                padding: 30px;
                text-align: center;
                border-bottom: 4px solid #F8961E;
            }}
            .header h1 {{
                font-size: 28px;
                margin-bottom: 10px;
                letter-spacing: 2px;
            }}
            .header h3 {{
                font-size: 16px;
                font-weight: normal;
                opacity: 0.8;
            }}
            .fecha {{
                margin-top: 15px;
                font-size: 12px;
                opacity: 0.7;
            }}
            /* INFO EQUIPO */
            .team-info {{
                background: #f8f9fa;
                padding: 20px 30px;
                border-bottom: 1px solid #e0e0e0;
                display: flex;
                justify-content: space-between;
                flex-wrap: wrap;
                gap: 20px;
            }}
            .team-card {{
                background: white;
                padding: 15px 25px;
                border-radius: 12px;
                box-shadow: 0 2px 8px rgba(0,0,0,0.1);
                flex: 1;
                min-width: 200px;
            }}
            .team-card h4 {{
                color: #4472C4;
                margin-bottom: 10px;
                font-size: 14px;
                text-transform: uppercase;
                letter-spacing: 1px;
            }}
            .team-card p, .team-card li {{
                font-size: 13px;
                color: #333;
            }}
            .team-card ul {{
                list-style: none;
                padding-left: 0;
            }}
            .team-card li {{
                padding: 3px 0;
            }}
            /* KPIs */
            .kpi-section {{
                background: #1a1a2e;
                padding: 25px;
                color: white;
            }}
            .kpi-grid {{
                display: flex;
                justify-content: space-around;
                flex-wrap: wrap;
                gap: 15px;
            }}
            .kpi-card {{
                text-align: center;
                background: rgba(255,255,255,0.1);
                padding: 15px 25px;
                border-radius: 12px;
                min-width: 150px;
                backdrop-filter: blur(5px);
            }}
            .kpi-card p {{
                font-size: 11px;
                opacity: 0.7;
                text-transform: uppercase;
                letter-spacing: 1px;
                margin-bottom: 8px;
            }}
            .kpi-card h2 {{
                font-size: 24px;
                margin: 0;
            }}
            .kpi-card .small {{
                font-size: 14px;
            }}
            /* CONTENIDO */
            .content {{
                padding: 30px;
            }}
            .insights {{
                background: #f0f2f5;
                padding: 20px;
                border-radius: 12px;
                margin-bottom: 30px;
            }}
            .insights h3 {{
                color: #1a1a2e;
                margin-bottom: 15px;
                border-left: 4px solid #F8961E;
                padding-left: 15px;
            }}
            .two-columns {{
                display: flex;
                gap: 20px;
                flex-wrap: wrap;
                margin-bottom: 25px;
            }}
            .column {{
                flex: 1;
                background: #f8f9fa;
                padding: 20px;
                border-radius: 12px;
            }}
            .column h4 {{
                color: #4472C4;
                margin-bottom: 12px;
                font-size: 16px;
            }}
            .column ul, .column ol {{
                padding-left: 20px;
            }}
            .column li {{
                margin: 8px 0;
                line-height: 1.5;
            }}
            .decisiones {{
                background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
                color: white;
                padding: 25px;
                border-radius: 12px;
                margin-top: 20px;
            }}
            .decisiones h3 {{
                margin-bottom: 20px;
                font-size: 20px;
            }}
            .decisiones li {{
                margin: 12px 0;
                line-height: 1.6;
            }}
            .footer {{
                background: #1a1a2e;
                color: white;
                text-align: center;
                padding: 20px;
                font-size: 12px;
                opacity: 0.7;
            }}
            hr {{
                margin: 20px 0;
                border: none;
                border-top: 1px solid #ddd;
            }}
            .badge {{
                display: inline-block;
                background: #4CAF50;
                color: white;
                font-size: 10px;
                padding: 2px 8px;
                border-radius: 20px;
                margin-left: 8px;
            }}
        </style>
    </head>
    <body>
        <div class="report-container">
            <!-- HEADER -->
            <div class="header">
                <h1>☕ CAFÉ MONTELOMA, S.A. de C.V.</h1>
                <h3>Reporte Ejecutivo Integrador - Análisis Comercial, Financiero y Operativo</h3>
                <div class="fecha">📅 Generado: {fecha_generacion}</div>
            </div>

            <!-- INFO EQUIPO -->
            <div class="team-info">
                <div class="team-card">
                    <h4>👥 EQUIPO DE TRABAJO</h4>
                    <ul>
                        {miembros_html}
                    </ul>
                </div>
                <div class="team-card">
                    <h4>🎓 ACADEMIA</h4>
                    <p><strong>Profesor:</strong> {profesor}</p>
                    <p><strong>Materia:</strong> {materia}</p>
                    <p><strong>Grupo:</strong> {grupo}</p>
                </div>
                <div class="team-card">
                    <h4>📊 FILTROS APLICADOS</h4>
                    <p><strong>Regiones:</strong> {', '.join(filtros_ejec_seleccionados['region'])}</p>
                    <p><strong>Canales:</strong> {', '.join(filtros_ejec_seleccionados['canal'])}</p>
                    <p><strong>Segmentos:</strong> {', '.join(filtros_ejec_seleccionados['segmento'])}</p>
                </div>
            </div>

            <!-- KPIs -->
            <div class="kpi-section">
                <div class="kpi-grid">
                    <div class="kpi-card">
                        <p>💰 VENTAS NETAS</p>
                        <h2>${ventas_netas:,.0f}</h2>
                    </div>
                    <div class="kpi-card">
                        <p>📈 MARGEN TOTAL</p>
                        <h2>${margen_total:,.0f} <span class="small">({margen_pct:.1%})</span></h2>
                    </div>
                    <div class="kpi-card">
                        <p>🎯 CUMPLIMIENTO</p>
                        <h2>{cumplimiento_pct:.1%}</h2>
                    </div>
                    <div class="kpi-card">
                        <p>🏷️ DESCUENTO PROM.</p>
                        <h2>{descuento_prom:.1%}</h2>
                    </div>
                    <div class="kpi-card">
                        <p>📦 DEVOLUCIÓN</p>
                        <h2>{devolucion_pct:.1%}</h2>
                    </div>
                </div>
            </div>

            <!-- INSIGHTS -->
            <div class="content">
                <div class="insights">
                    <h3>📌 HALLazgos CLAVE DEL ANÁLISIS</h3>
                    <div class="two-columns">
                        <div class="column">
                            <h4>✅ OPORTUNIDADES COMERCIALES</h4>
                            <ul>
                                <li><strong>{mejor_familia}</strong> - Familia de producto más rentable <span class="badge">Prioridad</span></li>
                                <li><strong>{mejor_canal}</strong> - Canal con mejor margen de contribución</li>
                                <li><strong>{mejor_region}</strong> - Región con mayor cumplimiento de presupuesto</li>
                                <li>Margen global <strong>{margen_pct:.1%}</strong> - {'Supera' if margen_pct >= 0.28 else 'Requiere atención'} el objetivo del 28%</li>
                            </ul>
                        </div>
                        <div class="column">
                            <h4>⚠️ ÁREAS DE OPORTUNIDAD (FOCO ROJO)</h4>
                            <ul>
                                <li><strong>{peor_familia}</strong> - Familia con menor rentabilidad</li>
                                <li><strong>{peor_canal}</strong> - Canal que requiere revisión de estrategia</li>
                                <li>Descuento promedio del <strong>{descuento_prom:.1%}</strong> - Evaluar impacto en margen</li>
                                <li>Tasa de devolución: <strong>{devolucion_pct:.1%}</strong></li>
                            </ul>
                        </div>
                    </div>
                </div>

                <!-- DECISIONES -->
                <div class="decisiones">
                    <h3>🎯 3 DECISIONES ESTRATÉGICAS PARA EL SIGUIENTE TRIMESTRE</h3>
                    <ol>
                        <li><strong>Invertir en {mejor_familia}:</strong> Aumentar presupuesto de producción y marketing para la familia más rentable, aprovechando su alto margen de contribución.</li>
                        <li><strong>Revisar estrategia en {peor_canal}:</strong> Evaluar política de descuentos, optimizar costos de distribución y considerar renegociación de condiciones comerciales.</li>
                        <li><strong>Enfocar esfuerzos en {mejor_region}:</strong> Duplicar inversión de marketing y fortalecer alianzas estratégicas en la región con mejor desempeño.</li>
                    </ol>
                </div>

                <hr>

                <div style="text-align: center; font-size: 12px; color: #666;">
                    <p>Reporte generado automáticamente por Macro de Automatización - Café Monteloma</p>
                    <p>Los datos presentados son basados en la información disponible del período analizado.</p>
                </div>
            </div>

            <!-- FOOTER -->
            <div class="footer">
                <p>Confidencial - Uso exclusivo para toma de decisiones internas</p>
                <p>© 2025 Café Monteloma S.A. de C.V. - Todos los derechos reservados</p>
            </div>
        </div>
    </body>
    </html>
    """

    # Guardar el reporte como archivo HTML
    with open('Reporte_Ejecutivo_Cafe_Monteloma.html', 'w', encoding='utf-8') as f:
        f.write(html_reporte)

    print("\n" + "=" * 70)
    print("✅ REPORTE GENERADO EXITOSAMENTE")
    print("=" * 70)
    print(f"📄 Archivo: 'Reporte_Ejecutivo_Cafe_Monteloma.html'")
    print(f"📅 Fecha: {fecha_generacion}")
    print(f"👥 Equipo: {', '.join(miembros_equipo[:3])} y {len(miembros_equipo)-3} más")
    print("=" * 70)
    print("\n📌 INSTRUCCIONES:")
    print("   1. Haz clic en el ícono de archivos 📁 en el panel izquierdo de Colab")
    print("   2. Busca el archivo 'Reporte_Ejecutivo_Cafe_Monteloma.html'")
    print("   3. Haz clic derecho y selecciona 'Descargar'")
    print("   4. Ábrelo en tu navegador para ver el reporte formateado")
    print("   5. Desde el navegador, puedes imprimirlo como PDF (Ctrl+P → Guardar como PDF)")
    print("\n" + "=" * 70)

    # Mostrar una vista previa del reporte en la celda
    print("\n📺 VISTA PREVIA DEL REPORTE:")
    display(HTML(html_reporte))

# Botón para ejecutar la macro
btn_reporte = widgets.Button(
    description="📄 GENERAR REPORTE EJECUTIVO (MACRO)",
    button_style='success',
    layout=widgets.Layout(width='100%', padding='10px', margin='10px 0px')
)

btn_reporte.on_click(generar_reporte_ejecutivo)

# Mostrar el botón
display(HTML("<h2>🤖 MACRO DE AUTOMATIZACIÓN</h2>"))
display(HTML("<p>Haz clic en el botón para generar el Reporte Ejecutivo completo con los datos actuales:</p>"))
display(btn_reporte)
print("\n✅ Macro lista para usar. Haz clic en el botón azul/verde para generar el reporte.")

📄 MACRO: GENERANDO REPORTE EJECUTIVO

✅ REPORTE GENERADO EXITOSAMENTE
📄 Archivo: 'Reporte_Ejecutivo_Cafe_Monteloma.html'
📅 Fecha: 08 de April de 2026 - 03:43 hrs
👥 Equipo: Almanza Frausto Eduardo Jose Francisco, Juan Diego Argüello Nájera, Hernandez Hernandez Abilene y 2 más

📌 INSTRUCCIONES:
   1. Haz clic en el ícono de archivos 📁 en el panel izquierdo de Colab
   2. Busca el archivo 'Reporte_Ejecutivo_Cafe_Monteloma.html'
   3. Haz clic derecho y selecciona 'Descargar'
   4. Ábrelo en tu navegador para ver el reporte formateado
   5. Desde el navegador, puedes imprimirlo como PDF (Ctrl+P → Guardar como PDF)


📺 VISTA PREVIA DEL REPORTE:
